In [1]:
import os
import pandas as pd
from glob import glob
# ! pip install chardet

In [100]:
def get_filetered_csv(csv_file):
    _df = pd.read_csv(csv_file)
    _df = _df.loc[:, ~_df.columns.str.contains('^Unnamed')]
    _df.reset_index(drop=True, inplace=True)

    _df = _df[_df['발생지시도'] == '서울']
    # 이게 [발생년월일시분] or [발생년월일시, 발생분] 
    if '발생년월일시분' in _df.columns:
        # 2015-01-01 05:57 형시
        _df['year'] = _df['발생년월일시분'].str.split(' ').str[0].str.split('-').str[0]
        _df['month'] = _df['발생년월일시분'].str.split(' ').str[0].str.split('-').str[1]
        _df['day'] = _df['발생년월일시분'].str.split(' ').str[0].str.split('-').str[2]
        _df['hour'] = _df['발생년월일시분'].str.split(' ').str[1].str.split(':').str[0].str.zfill(2)
        # _df['minute'] = _df['발생년월일시분'].str.split(' ').str[1].str.split(':').str[1]  
    else:
        _df['year'] = _df['발생년월일시'].str.split(' ').str[0].str.split('-').str[0]
        _df['month'] = _df['발생년월일시'].str.split(' ').str[0].str.split('-').str[1]
        _df['day'] = _df['발생년월일시'].str.split(' ').str[0].str.split('-').str[2]
        _df['hour'] = _df['발생년월일시'].str.split(' ').str[1].str.split(':').str[0].str.zfill(2)
    
    _df = _df[['year', 'month','day', 'hour', '주야', '요일', '발생지시도', '발생지시군구', '사고유형_대분류', '경도', '위도']]
    _df["datetime"] = pd.to_datetime(
        _df["year"] + _df["month"] + _df["day"] + _df["hour"],
        format="%Y%m%d%H"
    )
    
    # _df["발생일시"] = pd.to_datetime(_df["발생년월일시"], format="%Y-%m-%d %H")
    # _df = _df.rename(columns={'경도' : 'lon', '위도': 'lat'})
    # 경도 나 위도가 NaN인 경우는 삭제
    _df = _df[~_df['경도'].isna()]
    _df = _df[~_df['위도'].isna()]
    return _df

root= '../files2/origin'
file_list = sorted(glob(f"{root}/*_data.csv"))
df_list = [get_filetered_csv(f) for f in file_list]

df_all = pd.concat(df_list, axis=0)
df_all.reset_index(drop=True, inplace=True)
df_all = df_all.rename(columns={
    '경도' : 'lon', 
    '위도': 'lat',
    "발생지시도":      "city",
    "발생지시군구":    "district",
    "사고유형_대분류":"accident_type",
    '요일' : 'day_of_week',
    '주야' : 'day_night',
})
# df_all = df_all.set_index("datetime").sort_index()
df_all

,year,month,day,hour,day_night,day_of_week,city,district,accident_type,lon,lat,datetime
0,2012,01,13,16,주간,화,서울,강서구,차량단독,126.821995,37.544155,2012-01-13 16:00:00
1,2012,07,09,10,주간,월,서울,서초구,차대차,127.051443,37.451040,2012-07-09 10:00:00
2,2012,01,01,01,야간,일,서울,은평구,차대사람,126.931877,37.612850,2012-01-01 01:00:00
3,2012,08,28,04,야간,화,서울,서초구,차대차,127.051336,37.451815,2012-08-28 04:00:00
4,2012,01,07,23,야간,일,서울,강남구,차대차,127.050853,37.500513,2012-01-07 23:00:00
...,...,...,...,...,...,...,...,...,...,...,...,...
3614,2023,12,16,10,주,토,서울,마포구,차대사람,126.948712,37.540939,2023-12-16 10:00:00
3615,2023,12,22,08,주,금,서울,동대문구,차대사람,127.065072,37.570917,2023-12-22 08:00:00
3616,2023,12,27,05,야,수,서울,마포구,차대차,126.949564,37.542052,2023-12-27 05:00:00
3617,2023,12,27,10,주,수,서울,중구,차대사람,126.982910,37.568522,2023-12-27 10:00:00


In [101]:
import geopandas as gpd
from shapely.geometry import Point

gdf = gpd.GeoDataFrame(
    df_all,
    geometry=[Point(xy) for xy in zip(df_all.lon, df_all.lat)],
    crs="EPSG:4326"
)
gdf.head()

,year,month,day,hour,day_night,day_of_week,city,district,accident_type,lon,lat,datetime,geometry
0,2012,01,13,16,주간,화,서울,강서구,차량단독,126.821995,37.544155,2012-01-13 16:00:00,POINT (126.82199 37.54416)
1,2012,07,09,10,주간,월,서울,서초구,차대차,127.051443,37.451040,2012-07-09 10:00:00,POINT (127.05144 37.45104)
2,2012,01,01,01,야간,일,서울,은평구,차대사람,126.931877,37.612850,2012-01-01 01:00:00,POINT (126.93188 37.61285)
3,2012,08,28,04,야간,화,서울,서초구,차대차,127.051336,37.451815,2012-08-28 04:00:00,POINT (127.05134 37.45181)
4,2012,01,07,23,야간,일,서울,강남구,차대차,127.050853,37.500513,2012-01-07 23:00:00,POINT (127.05085 37.50051)


In [102]:
df_all.to_csv('../files2/final/seoul_accidents.csv', index=False)

---

### (2) 보행등 

In [19]:
import chardet
def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        result = chardet.detect(f.read())
        print(result['encoding'])  # 인코딩 확인
    df = pd.read_csv(file_path, encoding=result['encoding'])
    df.to_csv(file_path)

csv_roof = '../files2/origin'
# detect_encoding(f"{csv_roof}/seoul_traffic_lights.csv")
df_light = pd.read_csv(f"{csv_roof}/seoul_traffic_lights.csv")
df_light = df_light.loc[:, ~df_light.columns.str.contains('^Unnamed')]
df_light = df_light[['자치구', '신호등종류', '위도', '경도', '주소']]
df_light = df_light.rename(columns={
    '자치구' : 'district',
    '신호등종류' : 'light_type',
    '위도' : 'lat',
    '경도' : 'lon',
    '주소' : 'address'
})
df_light.to_csv(f'{csv_roof}/seoul_traffic_lights.csv', index=False)
df_light

,district,light_type,lat,lon,address
0,강남구,보행등,37.498560,127.044475,강남구 역삼동 710 대
1,강남구,보행등,37.509105,127.033164,강남구 논현동 219-12 대
2,강남구,보행등,37.495683,127.085581,강남구 일원동 460천
3,강남구,보행등,37.495809,127.085987,강남구 일원동 460천
4,강남구,보행등,37.534857,127.034393,강남구 압구정동 484천
...,...,...,...,...,...
24133,중랑구,보행등,37.616689,127.079436,중랑구 묵동 649-4대
24134,중랑구,보행등,37.616689,127.079436,중랑구 묵동 649-4대
24135,중랑구,보행등,37.614251,127.091812,중랑구 신내동 640 차
24136,중랑구,보행등,37.616534,127.079397,중랑구 묵동 656-2도
